# IBM HR Employee Attrition Analysis

**Author**: JosteinCT  
**Date Created**: September 29, 2026  
**Dataset**: [Kaggle - IBM HR Analytics](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset)  

---

## Executive Summary

*This project explores employee attrition patterns using exploratory data analysis (EDA) techniques in Python. The goal is to identify key factors driving employee turnover and provide actionable retention recommendations for HR leadership.*

## Business Questions

1. What factors correlate most strongly with employee attrition?
2. Which departments have highest/lowest turnover rates?
3. Is there a relationship between compensation and attrition risk?
4. How do tenure, age, and job satisfaction interact with attrition?
5. What retention strategies would you recommend based on the findings?

---

## 1. Import Libraries

*Loading required Python libraries for data manipulation, statistical analysis, and visualization.*

In [79]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

# Set random seed for reproducibility
np.random.seed(42)

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set style for visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


---

## 2. Load & Inspect Messy Dataset

*Before any cleaning, I loaded the raw dataset with intentional data quality issues. This allows me to document the problems systematically and demonstrate my cleaning methodology.*

In [80]:
# Define file paths
MESSY_PATH = '../data/raw/IBM_HR_Employee_Attrition_MESSY.csv'
CLEAN_PATH = '../data/cleaned/IBM_HR_Employee_Attrition_CLEAN.csv'

# Load messy dataset
df_raw = pd.read_csv(MESSY_PATH)

print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"\nFirst 5 rows:")
display(df_raw.head())

Shape: 1,520 rows × 36 columns

First 5 rows:


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,EmployeeName
0,41.0,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4.0,Single,NaN,19479,8,Y,yes,11,3,1,80,0,8,0,1,6,4,0,5,Employee_0
1,49.0,No,Travel_Frequently,279,Research & Development,8,1,NaN,1,2,3,Male,61,2,2,Research Scientist,2.0,Married,5130.0,24907,1,Y,NO,23,4,4,80,1,10,3,3,10,7,1,7,Employee_1
2,37.0,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3.0,Single,2090.0,2396,6,Y,yes,15,3,2,80,0,7,3,3,0,0,0,0,Employee_2
3,33.0,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3.0,Married,2909.0,23159,1,Y,yes,11,3,3,80,0,8,3,3,8,7,3,0,Employee_3
4,27.0,No,travel_rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2.0,Married,3468.0,16632,9,Y,NO,12,3,4,80,1,6,3,3,2,2,2,2,Employee_4


In [81]:
import os

print(f"File Path Being Loaded: {MESSY_PATH}")
print(f"Full Resolved Path: {os.path.abspath(MESSY_PATH)}")
print(f"\nDoes file exist? {os.path.exists(MESSY_PATH)}")
print(f"File size: {os.path.getsize(MESSY_PATH)/1024:.2f} KB")

# Compare with clean file
print(f"\n--- File Comparison ---")
print(f"MESSY file exists? {os.path.exists(MESSY_PATH)}")
print(f"CLEAN file exists? {os.path.exists(CLEAN_PATH)}")

# Check row counts from filesystem
messy_df_temp = pd.read_csv(MESSY_PATH)
clean_df_temp = pd.read_csv(CLEAN_PATH.replace('MESSY', 'CLEAN'))
print(f"\nMessy CSV rows: {len(messy_df_temp):,}")
print(f"Clean CSV rows: {len(clean_df_temp):,}")

File Path Being Loaded: ../data/raw/IBM_HR_Employee_Attrition_MESSY.csv
Full Resolved Path: c:\Users\j384\Documents\portfolio\project-2-python\data\raw\IBM_HR_Employee_Attrition_MESSY.csv

Does file exist? True
File size: 256.00 KB

--- File Comparison ---
MESSY file exists? True
CLEAN file exists? True

Messy CSV rows: 1,520
Clean CSV rows: 1,470


*Initial inspection reveals the dataset structure. Now I'll profile the data quality issues systematically.*

In [82]:
# Quick overview using built-in method
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1520 entries, 0 to 1519
Data columns (total 36 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       1434 non-null   float64
 1   Attrition                 1520 non-null   object 
 2   BusinessTravel            1520 non-null   object 
 3   DailyRate                 1520 non-null   int64  
 4   Department                1447 non-null   object 
 5   DistanceFromHome          1520 non-null   int64  
 6   Education                 1520 non-null   int64  
 7   EducationField            1454 non-null   object 
 8   EmployeeCount             1520 non-null   int64  
 9   EmployeeNumber            1520 non-null   int64  
 10  EnvironmentSatisfaction   1520 non-null   int64  
 11  Gender                    1520 non-null   object 
 12  HourlyRate                1520 non-null   int64  
 13  JobInvolvement            1520 non-null   int64  
 14  JobLevel

In [83]:
# Additional breakdown by data type
numeric_cols = df_raw.select_dtypes(include=['number']).columns.tolist()
cat_cols = df_raw.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumerical columns ({len(numeric_cols)}): {numeric_cols[:5]}...")
print(f"Categorical columns ({len(cat_cols)}): {cat_cols[:5]}...")


Numerical columns (25): ['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EmployeeCount']...
Categorical columns (11): ['Attrition', 'BusinessTravel', 'Department', 'EducationField', 'Gender']...


### Missing Value Analysis

*Checking for null values across all columns. Expected: ~5% missing in Age, MonthlyIncome, JobSatisfaction, EducationField, and Department.*

In [84]:
# Check for missing values
missing_data = df_raw.isnull().sum()
missing_percentage = (missing_data / len(df_raw)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_data,
    'Missing %': missing_percentage.round(2)
}).sort_values('Missing Count', ascending=False)

print("MISSING VALUES ANALYSIS")
print("-"*40)
print(missing_df[missing_df['Missing Count'] > 0])

MISSING VALUES ANALYSIS
----------------------------------------
                 Missing Count  Missing %
MonthlyIncome              104       6.84
Age                         86       5.66
Department                  73       4.80
JobSatisfaction             67       4.41
EducationField              66       4.34
EmployeeName                 1       0.07


### Duplicate Record Detection

*Checking for exact duplicate rows. Expected: 50 synthetic duplicates added via inject_issues.py script.*

In [85]:
# Check for duplicates
duplicate_count = df_raw.duplicated().sum()
print(f"DUPLICATE RECORDS DETECTED: {duplicate_count:,}")

# Show sample duplicate rows (if any)
if duplicate_count > 0:
    print("\nSample duplicate row:")
    print(df_raw[df_raw.duplicated()].iloc[0])

DUPLICATE RECORDS DETECTED: 0


In [86]:
# More thorough duplicate check
print("CHECKING FOR DUPLICATES:")
print("-"*40)

# Method 1: Exact row duplicates
exact_dupes = df_raw.duplicated().sum()
print(f"Exact duplicates (all columns): {exact_dupes:,}")

# Method 2: Partial duplicates (key columns only)
key_cols = ['EmployeeNumber', 'Age', 'MonthlyIncome', 'Department']
partial_dupes = df_raw.duplicated(subset=key_cols).sum()
print(f"Partial duplicates (key columns): {partial_dupes:,}")

# Method 3: Check row count vs original
print(f"\nOriginal expected rows (Kaggle): ~1,470")
print(f"Current row count: {len(df_raw):,}")
print(f"Row difference: {len(df_raw) - 1470:,}")

CHECKING FOR DUPLICATES:
----------------------------------------
Exact duplicates (all columns): 0
Partial duplicates (key columns): 48

Original expected rows (Kaggle): ~1,470
Current row count: 1,520
Row difference: 50


In [87]:
# Verify specific data quality issues
print("DATA QUALITY VERIFICATION:")
print("-"*40)

# 1. Check for missing values
total_missing = df_raw.isnull().sum().sum()
print(f"1. Total missing values: {total_missing:,} (expected: ~375)")

# 2. Check MonthlyIncome for non-numeric text (FIXED VERSION)
income_non_numeric = pd.to_numeric(df_raw['MonthlyIncome'], errors='coerce').isna().sum()
print(f"2. Non-numeric MonthlyIncome values: {income_non_numeric:,} (expected: 30)")

# 3. Check Age outliers
age_outliers = ((df_raw['Age'] < 18) | (df_raw['Age'] > 70)).sum()
print(f"3. Age outliers (<18 or >70): {age_outliers:,} (expected: 10)")

# 4. Check YearsAtCompany outliers
tenure_outliers = (df_raw['YearsAtCompany'] > 40).sum()
print(f"4. Tenure outliers (>40 years): {tenure_outliers:,} (expected: 8)")

# 5. Check inconsistent casing
print(f"\n5. BusinessTravel unique values: {sorted(df_raw['BusinessTravel'].unique())}")
print(f"6. OverTime unique values: {sorted(df_raw['OverTime'].unique())}")

DATA QUALITY VERIFICATION:
----------------------------------------
1. Total missing values: 397 (expected: ~375)
2. Non-numeric MonthlyIncome values: 110 (expected: 30)
3. Age outliers (<18 or >70): 10 (expected: 10)
4. Tenure outliers (>40 years): 8 (expected: 8)

5. BusinessTravel unique values: ['Non-Travel', 'Travel_Frequently', 'Travel_Rarely', 'non-travel', 'travel_frequently', 'travel_rarely']
6. OverTime unique values: ['NO', 'yes']
